MD# ADVANCE NATURAL LANGUAGE PROCESSING (AML23702) - Lab Exercise 5

**Name:** P Prithvi Raj Kandan &nbsp;|&nbsp; **USN:** 1GA23AI033 &nbsp;|&nbsp; Dept. of AI & ML, Global Academy of Technology

**Aim:** Use NLTK WordNet to find the synonyms and definitions of a word and distinguish its meanings in different contexts (Lesk algorithm).

In [1]:
%pip install -q nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
from pathlib import Path

def ensure(resource, package):
    """Download an NLTK resource only if it is not already installed."""
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True)
ensure("corpora/wordnet", "wordnet")
ensure("corpora/omw-1.4", "omw-1.4")
ensure("tokenizers/punkt_tab", "punkt_tab")

from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

DATA = Path("../data")

def load_lines(name):
    return [ln.strip() for ln in (DATA / name).read_text(encoding="utf-8").splitlines() if ln.strip()]


In [3]:
def explore_word(word):
    synsets = wn.synsets(word)

    if not synsets:
        print("No WordNet entries found for:", word)
        return

    print("Word:", word)
    print("\nPossible meanings:")

    for i, synset in enumerate(synsets, start=1):
        synonyms = sorted({
            lemma.name().replace("_", " ")
            for lemma in synset.lemmas()
        })
        print(f"\n{i}. Synset: {synset.name()}")
        print("   Definition:", synset.definition())
        print("   Synonyms:", ", ".join(synonyms))

explore_word((DATA / "bank_word.txt").read_text(encoding="utf-8").strip())


Word: bank

Possible meanings:

1. Synset: bank.n.01
   Definition: sloping land (especially the slope beside a body of water)
   Synonyms: bank

2. Synset: depository_financial_institution.n.01
   Definition: a financial institution that accepts deposits and channels the money into lending activities
   Synonyms: bank, banking company, banking concern, depository financial institution

3. Synset: bank.n.03
   Definition: a long ridge or pile
   Synonyms: bank

4. Synset: bank.n.04
   Definition: an arrangement of similar objects in a row or in tiers
   Synonyms: bank

5. Synset: bank.n.05
   Definition: a supply or stock held in reserve for future use (especially in emergencies)
   Synonyms: bank

6. Synset: bank.n.06
   Definition: the funds held by a gambling house or the dealer in some gambling games
   Synonyms: bank

7. Synset: bank.n.07
   Definition: a slope in the turn of a road or track; the outside is higher than the inside in order to reduce the effects of centrifugal force

In [4]:
# Context-based word-sense identification with the Lesk algorithm.
word = (DATA / "bank_word.txt").read_text(encoding="utf-8").strip()
sentences = load_lines("bank_contexts.txt")

for sentence in sentences:
    sense = lesk(sentence.split(), word)
    print("\nSentence:", sentence)

    if sense:
        print("Predicted meaning:", sense.definition())
        print("Synonyms:", ", ".join(
            lemma.name().replace("_", " ") for lemma in sense.lemmas()
        ))
    else:
        print("No sense could be selected.")



Sentence: She deposited her money at the bank.
Predicted meaning: a container (usually with a slot in the top) for keeping money at home
Synonyms: savings bank, coin bank, money box, bank

Sentence: They sat on the bank of the river.
Predicted meaning: sloping land (especially the slope beside a body of water)
Synonyms: bank


MD### Second word: *spring* (used by the web frontend)

In [5]:
explore_word((DATA / "word.txt").read_text(encoding="utf-8").strip())


Word: spring

Possible meanings:

1. Synset: spring.n.01
   Definition: the season of growth
   Synonyms: spring, springtime

2. Synset: spring.n.02
   Definition: a metal elastic device that returns to its shape or position when pushed or pulled or pressed
   Synonyms: spring

3. Synset: spring.n.03
   Definition: a natural flow of ground water
   Synonyms: fountain, natural spring, outflow, outpouring, spring

4. Synset: spring.n.04
   Definition: a point at which water issues forth
   Synonyms: spring

5. Synset: give.n.01
   Definition: the elasticity of something that can be stretched and returns to its original length
   Synonyms: give, spring, springiness

6. Synset: leap.n.01
   Definition: a light, self-propelled movement upwards or forwards
   Synonyms: bounce, bound, leap, leaping, saltation, spring

7. Synset: jump.v.01
   Definition: move forward by leaps and bounds
   Synonyms: bound, jump, leap, spring

8. Synset: form.v.03
   Definition: develop into a distinctive entit

In [6]:
word = (DATA / "word.txt").read_text(encoding="utf-8").strip()
contexts = load_lines("contexts.txt")
for sentence in contexts:
    sense = lesk(nltk.word_tokenize(sentence), word)
    print(f"{sentence}\n   -> {sense.name()}: {sense.definition()}\n")


Flowers bloom early in the spring season.
   -> spring.n.01: the season of growth

The old mattress had a broken metal spring.
   -> spring.n.02: a metal elastic device that returns to its shape or position when pushed or pulled or pressed

We drank cold water from a mountain spring.
   -> spring.n.03: a natural flow of ground water



In [7]:
# A case where simplified Lesk fails: both sentences get the music sense
word = (DATA / "bass_word.txt").read_text(encoding="utf-8").strip()
for sentence in load_lines("bass_contexts.txt"):
    sense = lesk(nltk.word_tokenize(sentence), word)
    print(f"{sentence}\n   -> {sense.name()}: {sense.definition()}\n")


He plays the bass guitar in a rock band.
   -> bass.n.02: the lowest part in polyphonic music

We caught a large bass while fishing in the lake.
   -> bass.n.02: the lowest part in polyphonic music



MD Lesk picks the sense whose gloss shares the most words with the sentence. The *spring* sentences each overlap with the right gloss. For *bass*, the fishing sentence shares no words with the fish glosses, so Lesk falls back to a music sense, which is wrong.